In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

splits_root    = Path("splits")
panel_filtered = Path("processed_coords_30thOct_joined_new_mapped_corrected_panel_QC")
patches_root   = Path("patches"); patches_root.mkdir(exist_ok=True)

PATCH_SIZE = 224
HALF       = PATCH_SIZE // 2

meta = pd.read_csv(
    "Human breast cancer in situ capturing transcriptomics/metadata.csv")
meta.columns    = meta.columns.str.strip()
meta["patient"] = meta["count_matrix"].str.extract(r"(\d{5})").astype(int)

raw = Path("Human breast cancer in situ capturing transcriptomics")

index_records = []

for _, row in meta.iterrows():
    patient, section = row["patient"], row["replicate"]

    # --- load spot table for this section ---
    spot_file = panel_filtered / f"{patient}_{section}.csv.gz"
    if not spot_file.exists():
        print(f"SKIP {patient}_{section}: spot file not found"); continue
    spots = pd.read_csv(spot_file, index_col=0)
    spots = spots.dropna(subset=["pixel_x", "pixel_y"])
    n     = len(spots)

    # --- load H&E image ---
    img_path = next(raw.rglob(row["histology_image"]), None)
    if img_path is None:
        print(f"SKIP {patient}_{section}: image not found"); continue
    img = np.array(Image.open(img_path).convert("RGB"))
    H, W = img.shape[:2]

    # --- create memmap ---
    mmap_path = patches_root / f"{patient}_{section}.npy"
    mmap      = np.memmap(mmap_path, dtype="uint8", mode="w+",
                          shape=(n, PATCH_SIZE, PATCH_SIZE, 3))

    # --- extract patches ---
    for idx, (spot_id, spot) in enumerate(spots.iterrows()):
        cx, cy = int(round(spot["pixel_x"])), int(round(spot["pixel_y"]))

        # clamp to image bounds
        x1 = max(cx - HALF, 0);  x2 = min(cx + HALF, W)
        y1 = max(cy - HALF, 0);  y2 = min(cy + HALF, H)
        crop = img[y1:y2, x1:x2]

        # pad if border spot
        patch = np.zeros((PATCH_SIZE, PATCH_SIZE, 3), dtype="uint8")
        ph    = crop.shape[0]; pw = crop.shape[1]
        patch[:ph, :pw] = crop
        mmap[idx]       = patch

        index_records.append(dict(
            patient   = patient,
            section   = section,
            spot_id   = spot_id,
            mmap_path = str(mmap_path),
            mmap_idx  = idx,
            pixel_x   = cx,
            pixel_y   = cy,
            tumor     = spot["tumor"]
        ))

    del mmap   # flush to disk
    print(f"{patient}_{section} | spots: {n} | image: {W}x{H}")

# --- save master patch index ---
index_df = pd.DataFrame(index_records)
index_df.to_csv(patches_root / "patch_index.csv", index=False)

print(f"\nDone. Total patches: {len(index_records)}")
print(f"Patch index saved to patches/patch_index.csv")